# 4.16 · 等张回归 / Isotonic Regression

> **课程定位 / Where this fits**
> 第 16 课，**Part 4 · 监督学习：回归**（收官）。
> Lesson 16, **Part 4 · Supervised Regression** (finale).
>
> 有时你**确信关系是单调的**（剂量越大反应越强、面积越大房价越高），但**不知道具体形状**（线性?S形?对数?）。**等张回归**只假设单调、不假设形状，拟合出一个**单调阶梯函数**。它最重要的工业用途是**概率校准**——把分类器"不准的概率"修正成"可信的概率"（接 5.15）。
> Sometimes you're **sure a relationship is monotonic** (more dose → stronger response, bigger area → higher price) but **don't know its shape** (linear? S-curve? log?). **Isotonic regression** assumes only monotonicity, no shape, fitting a **monotonic staircase**. Its key industrial use is **probability calibration** — turning a classifier's miscalibrated scores into trustworthy probabilities (see 5.15).
>
> 💼 **实战/面试视角**："怎么校准概率 / 单调约束怎么加" 偏风控/排序/校准场景。
> 💼 **Practical/interview angle:** "how to calibrate probabilities / impose monotonic constraints" — risk/ranking/calibration.

> 💡 **面试相关 / Interview-relevant**
> - "等张回归是什么 / PAV 算法"（出镜率 ★★★）
> - "概率校准 / 可靠性曲线"（★★★★，接 5.15）
> - "Isotonic vs Platt 校准"（★★★）
> - "什么时候用单调约束"（★★★）

---

## 学习目标 / Learning Objectives

1. 理解等张回归 = 只假设单调的非参数拟合。
   Understand isotonic regression = nonparametric fit assuming only monotonicity.
2. **从零**实现 PAV 算法，对照 sklearn。
   Implement the PAV algorithm from scratch, matching sklearn.
3. 对比线性/多项式/等张的取舍。
   Compare linear/polynomial/isotonic trade-offs.
4. 用等张回归做**概率校准**（最重要的用途）。
   Use isotonic regression for **probability calibration** (its key use).

## 目录 / TOC
1. [先建直觉 + 数据](#1)
2. [PAV 算法（从零）⭐](#2)
3. [线性 vs 多项式 vs 等张 ⭐](#3)
4. [杀手应用：概率校准 ⭐](#4)
5. [小结](#5)


<a id="1"></a>
## 1. 先建直觉 + 数据 / Intuition & Data

普通回归要么假设形状（线性=直线、多项式=曲线），要么完全自由（树、KNN）。等张回归在两者之间：它**只加一个约束——预测必须单调（非降）**，除此之外让数据自己说话。结果是一个**单调的阶梯函数**：能贴合任意单调形状（S 形、对数、分段），又保证"x 增大预测绝不减小"。
Ordinary regression either assumes a shape (linear/polynomial) or is fully free (trees, KNN). Isotonic regression sits between: it imposes **just one constraint — predictions must be monotonic (non-decreasing)** — and otherwise lets the data speak. The result is a **monotonic staircase**: it fits any monotonic shape (S-curve, log, piecewise) while guaranteeing "as x increases, the prediction never decreases".

造一份**真实是 S 形但我们假装不知道**的单调数据。
We make data that's truly an S-curve but we pretend not to know the shape.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.isotonic import IsotonicRegression
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

n = 80
x = np.sort(rng.uniform(0, 10, n))
y_true = 5 / (1 + np.exp(-(x - 5)))         # 真实是 S 形(单调递增)
y = y_true + rng.normal(0, 0.5, n)

# out_of_bounds='clip': 预测超出训练 x 范围时, 用边界值兜底 / clip out-of-range
iso = IsotonicRegression(out_of_bounds="clip").fit(x, y)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(x, y, alpha=0.4, s=20, label="数据(单调+噪声)")
ax.plot(x, y_true, "g--", lw=1.5, label="真实(S形)")
ax.plot(x, iso.predict(x), "r-", lw=2, label="等张回归(单调阶梯)")
ax.legend(); ax.set_title("等张回归: 只假设单调, 拟合出单调阶梯\n无需知道是 S形/线性/任何形式")
plt.tight_layout(); plt.show()
print("等张回归拟合出单调非减的阶梯, 自动贴合 S 形 — 完全没告诉它是 S 形")


<a id="2"></a>
## 2. PAV 算法（从零）⭐ / The PAV Algorithm

等张回归用 **PAV（Pool Adjacent Violators，合并相邻违反者）** 算法精确求解，思路极简：从左到右扫，一旦发现**相邻两点违反单调**（左 > 右），就把它们**合并成一个块、取加权平均**；合并后可能又和前面的块违反，于是回退继续合并，直到全局单调。它是 $O(n)$ 的精确算法。
Isotonic regression is solved exactly by **PAV (Pool Adjacent Violators)**, with a simple idea: scan left to right; whenever two adjacent points **violate monotonicity** (left > right), **pool them into one block and take their weighted average**; the merged block may now violate the previous one, so back up and keep merging until globally monotonic. An exact $O(n)$ algorithm.


In [ ]:
def pav(y):
    y = y.astype(float).copy()
    vals = list(y); weights = [1.0]*len(y)     # 每个块: 一个值 + 一个权重(块内点数)
    i = 0
    while i < len(vals) - 1:
        if vals[i] > vals[i+1]:                # 相邻违反单调(左>右) → 合并
            new_w = weights[i] + weights[i+1]
            new_v = (vals[i]*weights[i] + vals[i+1]*weights[i+1]) / new_w   # 加权平均
            vals[i] = new_v; weights[i] = new_w
            del vals[i+1]; del weights[i+1]
            if i > 0: i -= 1                    # 回退: 合并后可能与前一块又违反
        else:
            i += 1
    # 把每个块按其权重(点数)展开回原长度 / expand blocks back to length n
    result = []
    for v, w in zip(vals, weights):
        result.extend([v]*int(w))
    return np.array(result)

my_pav = pav(y)
sk_iso = IsotonicRegression().fit_transform(x, y)
print(f"从零 PAV vs sklearn 最大差异: {np.abs(my_pav - sk_iso).max():.6f} → 一致")
print(f"输出块数(阶梯段数): {len(np.unique(np.round(my_pav, 6)))} (原 {n} 点合并成几段)")
print("PAV 把违反单调的相邻点合并成加权平均, 直到全局单调 — O(n) 精确解")


<a id="3"></a>
## 3. 线性 vs 多项式 vs 等张 ⭐ / Linear vs Polynomial vs Isotonic

三种拟合单调数据的方式各有取舍：
Three ways to fit monotonic data, each with trade-offs:
- **线性**：太死板，强行用直线，拟合不了 S 形的弯曲。
  **Linear:** too rigid, forces a straight line, can't bend to the S-curve.
- **多项式**：灵活，但**可能违反单调**（高阶项让尾部翘起来，出现"x 增大但预测下降"）。
  **Polynomial:** flexible, but **may violate monotonicity** (high-order terms make tails curl up/down).
- **等张**：灵活贴合 + **保证单调**——当你确信单调、但不知形状时的最佳选择。
  **Isotonic:** flexible fit + **guaranteed monotonicity** — the best choice when you're sure it's monotonic but don't know the shape.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(x, y, alpha=0.35, s=20, label="数据 data")
xp = x.reshape(-1, 1)
ax.plot(x, LinearRegression().fit(xp, y).predict(xp), label="线性(假设直线)")
ax.plot(x, make_pipeline(PolynomialFeatures(3), LinearRegression()).fit(xp, y).predict(xp), label="3阶多项式(可能非单调)")
ax.plot(x, IsotonicRegression(out_of_bounds="clip").fit(x, y).predict(x), "r-", lw=2.5, label="等张(只假设单调)")
ax.legend(fontsize=9); ax.set_title("线性(太死板) vs 多项式(可能非单调) vs 等张(灵活且保证单调)")
plt.tight_layout(); plt.show()
print("线性: 太死板; 多项式: 灵活但可能违反单调(尾部翘起); 等张: 灵活贴合+保证单调")


<a id="4"></a>
## 4. 杀手应用：概率校准 ⭐ / Killer Use: Probability Calibration

等张回归最重要的工业用途。很多分类器（SVM、随机森林、boosting）输出的"概率"**排序对但数值不准**——比如它说 0.9，实际只有 70% 的把握（**过度自信**）。
Isotonic regression's most important industrial use. Many classifiers (SVM, random forests, boosting) output "probabilities" that are **correctly ranked but numerically wrong** — e.g. it says 0.9 when the true rate is 70% (**overconfident**).

校准的需求：单调（分数高的样本真实概率确实更高）但形状未知——**正好是等张回归的场景**。用它学一个"原始分数 → 真实概率"的单调映射，就能把分数校准成可信概率。**可靠性曲线(reliability diagram)** 验证：校准后曲线应贴合对角线（"说 0.9 就真有 90%"）。
The need: monotonic (higher-scored samples really do have higher true probability) but shape unknown — **exactly isotonic's setting**. It learns a monotonic "raw score → true probability" map, calibrating scores into trustworthy probabilities. The **reliability diagram** verifies: after calibration the curve should hug the diagonal ("0.9 really means 90%").


In [ ]:
from sklearn.calibration import calibration_curve

n2 = 2000
true_p = rng.uniform(0, 1, n2)
labels = (rng.uniform(0, 1, n2) < true_p).astype(int)     # 真实标签按 true_p 生成
# 一个"过度自信"的模型分数: 排序对但被推向 0/1 / overconfident but correctly-ranked scores
raw_score = 1/(1+np.exp(-6*(true_p-0.5)))

iso_cal = IsotonicRegression(out_of_bounds="clip").fit(raw_score, labels)   # 学 分数→真实概率 的单调映射
calibrated = iso_cal.predict(raw_score)

fig, ax = plt.subplots(figsize=(6, 5.5))
ax.plot([0,1],[0,1],"k--", label="完美校准 perfect")
for scores, name, c in [(raw_score,"原始分数(过度自信)","C3"), (calibrated,"等张校准后","C2")]:
    # calibration_curve: 把预测分箱, 看每箱的实际正类比例 vs 平均预测 / reliability diagram
    frac_pos, mean_pred = calibration_curve(labels, scores, n_bins=10, strategy="quantile")
    ax.plot(mean_pred, frac_pos, "o-", color=c, label=name)
ax.set_xlabel("预测概率 predicted prob"); ax.set_ylabel("实际正类比例 actual rate"); ax.legend()
ax.set_title("概率校准: 原始分数偏离对角线(不准); 等张校准后贴合对角线(可信)")
plt.tight_layout(); plt.show()
print("原始分数: 校准曲线偏离对角线(说0.9实际没那么高)=过度自信")
print("等张校准后: 贴合对角线 → '0.9 的预测里真有~90% 是正类'=概率可信")
print("💡 等张/Platt 校准是 Part 5.15 的正题; 这里展示等张回归最重要的工业用途")


<a id="5"></a>
## 5. 小结 / Summary

```
等张回归: 只假设单调(非降), 不假设形状 → 拟合单调阶梯函数
PAV 算法: 从左扫, 相邻违反单调就合并成加权平均块, 回退继续, 直到全局单调; O(n) 精确
vs 线性(死板)/多项式(可能违反单调): 等张灵活贴合 + 保证单调
杀手应用: 概率校准 — 学"分数→真实概率"的单调映射, 修正过度自信; 可靠性曲线贴对角线
Isotonic 校准灵活(需较多数据), Platt(sigmoid)校准参数少(小数据稳); 详见 5.15
```

### 💡 面试速查 / Interview cheat-sheet
1. **等张回归只假设单调**, 拟合单调阶梯(形状自由)。
   Isotonic assumes only monotonicity, fits a free-shaped monotonic staircase.
2. **PAV 算法**: 合并相邻违反者求精确解 O(n)。
   PAV: pool adjacent violators for the exact O(n) solution.
3. **最重要用途=概率校准**: 把不准的分数修成可信概率。
   Key use = probability calibration: fix miscalibrated scores into trustworthy probabilities.
4. **可靠性曲线**贴对角线 = 校准良好。
   A reliability curve on the diagonal = well-calibrated.
5. **Isotonic(灵活,需数据多) vs Platt(参数少,小数据稳)** 两种校准。
   Isotonic (flexible, data-hungry) vs Platt (few params, stable on small data).

### Part 4 完成 🎉
回归全部走通: 线性回归→诊断→多项式→正则(Ridge/Lasso/ElasticNet)→GLM→非线性→SVR/KNN→树/森林/GBDT/XGBoost→分位数/稳健/等张。下一部分 **Part 5 监督学习：分类**(已完成的章节)。
Part 4 complete: linear → diagnostics → polynomial → regularization → GLM → nonlinear → SVR/KNN → trees/forest/GBDT/XGBoost → quantile/robust/isotonic. Next, **Part 5 Classification**.
